# الدرس 10 - وكلاء الذكاء الاصطناعي في الإنتاج

في هذا الدرس ستتعلم **أنماط الإنتاج** لوكلاء الذكاء الاصطناعي باستخدام إطار عمل Microsoft Agent (بايثون). نغطي:

- **المراقبة** — إضافة توقيت وتسجيل لتفاعلات الوكيل
- **التقييم** — استخدام وكيل تقييم لتسجيل جودة الردود
- **إدارة التكلفة** — استراتيجيات لتحسين الرموز واختيار النموذج

السيناريو هو **وكيل سفر** يساعد المستخدمين في تخطيط الرحلات، مع المراقبة والتقييم مدمجين في الأعلى.


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## اعتبارات الإنتاج

يتطلب نقل وكلاء الذكاء الاصطناعي من النماذج الأولية إلى الإنتاج اهتمامًا دقيقًا بثلاثة أركان:

1. **المراقبة** — تحتاج إلى رؤية ما يفعله الوكيل، ومدة قيامه به، والأدوات التي يستدعيها. بدون التتبع وتسجيل الدخول، يكون تصحيح مشكلات الإنتاج أمرًا شبه مستحيل.

2. **التقييم** — تضمن عمليات الفحص الآلي جودة استجابات الوكيل وأن تظل دقيقة وكاملة ومفيدة مع مرور الوقت. يمكن لوكيل التقييم تقييم الاستجابات بناءً على معايير محددة.

3. **إدارة التكلفة** — يؤثر استخدام الرموز مباشرة على التكلفة. تساعد استراتيجيات مثل تحسين المطالبات، اختيار النموذج، والتخزين المؤقت في السيطرة على النفقات دون التضحية بالجودة.


## بناء وكيل قابل للمراقبة

نعرّف أدوات السفر ونلف استدعاء الوكيل مع التوقيت حتى نتمكن من مراقبة الكمون. في الإنتاج، ستدمج مع OpenTelemetry أو نظام تتبع مشابه.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## أنماط التقييم

نمط شائع في الإنتاج هو استخدام وكيل ثانٍ كـ **مقَيّم**. يقوم المقَيّم بتقييم استجابة الوكيل الأساسي مقابل معايير محددة مسبقًا مثل الشمول والدقة والفائدة.

يتيح ذلك:
- بوابات جودة مؤتمتة قبل وصول الاستجابات إلى المستخدمين
- الكشف عن التراجعات عند تغير التنبيهات أو النماذج
- المراقبة المستمرة لأداء الوكيل مع مرور الوقت


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## استراتيجيات إدارة التكاليف

يعد التحكم في التكاليف أمرًا حيويًا لوكلاء الذكاء الاصطناعي الإنتاجيين. إليك بعض الاستراتيجيات الرئيسية:

| الاستراتيجية | الوصف |
|---|---|
| **تحسين المطالبات** | اجعل تعليمات النظام مختصرة. أزل السياق الزائد لتقليل عدد التوكونات في الإدخال. |
    "| **اختيار النموذج** | استخدم نماذج أصغر وأرخص (مثل GPT-4.1-mini) للمهام البسيطة مثل التصنيف أو الاستخراج، واحتفظ بالنماذج الأكبر للمهام التي تتطلب تفكيرًا معقدًا. |\n",
| **التخزين المؤقت** | خزّن نتائج الأدوات والاستفسارات المتكررة لتجنب استدعاءات API غير الضرورية. |
| **ميزانيات التوكونات** | حدد حدود `max_tokens` لمنع الردود الطويلة غير المتوقعة. |
| **التجميع** | اجمع استفسارات المستخدمين المتعددة في استدعاء API واحد عند الإمكان. |

في الممارسة العملية، تعمل الطريقة الهرمية بشكل جيد: قم بتوجيه الطلبات البسيطة إلى نموذج سريع ورخيص وتصعيد الاستفسارات المعقدة فقط إلى نموذج أكثر قدرة.


## الملخص

في هذا الدرس تعلمت كيف:

1. **إضافة القابلية للملاحظة** لتفاعلات الوكيل باستخدام التوقيت والتسجيل، مما يمهد الطريق للتتبع والمراقبة.
2. **تقييم استجابات الوكيل** تلقائيًا باستخدام وكيل التقييم الذي يقوم بتسجيل الاكتمال والدقة والفائدة.
3. **إدارة التكاليف** من خلال تحسين التنبيهات، اختيار النموذج، التخزين المؤقت، وميزانيات الرموز.

تساعد هذه الأنماط الإنتاجية في ضمان أن وكلاء الذكاء الاصطناعي الخاصين بك موثوقون، قابلون للقياس وفعّالون من حيث التكلفة على نطاق واسع.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
